<a href="https://colab.research.google.com/github/SakshamPainuly/underwater-creature-tracking/blob/p4%2Fcolab-baseline/notebooks/colab_train_tritonia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UWCT — Colab YOLO26s training on Tritonia sea slugs (E006, dataset v3)

This notebook only calls repository scripts; every check lives in the code (D-029, D-030, D-031).

- Runtime: **GPU** (Runtime → Change runtime type). Never train on the local PC.
- Dataset **v3**: the 150-frame batch 1, class `1: sea_slug`, split by recording day (10 train /
  3 val / 3 test days, D-030). Its boxes were made by code next to each MTrackJ cross and not
  reviewed (D-031). The training data file has **no test split**.
- **Accuracy is measured against the crosses** (cell 17: point precision, recall, F1, point AP on
  val). The P/R/mAP that Ultralytics prints during training are against the automatic boxes:
  they are not accuracy and are never reported as such (D-031).
- Runs and checkpoints go to `MyDrive/uwct/runs/` (outside Git). Every cell raises on failure.
- Frames are 15 s apart: results are sparse-frame/timelapse analysis, never "video tracking".

**Upload to Google Drive first (once):**
1. `D:\uwct_raw\tritonia_boxes_batch1\` (the whole folder, made by `scripts/ingest_tritonia_boxes.py`)
   → `MyDrive/uwct/raw/tritonia_boxes_batch1/`
2. `D:\uwct_work\tritonia_label_batch1\` (the batch: `images/`, `points/`, `manifest.csv`,
   `batch.json`; already uploaded 2026-11-03) → `MyDrive/uwct/work/tritonia_label_batch1/`

**First session (no E006 config yet):** run cells 1–10. Cell 9 prints the dataset fingerprint;
put it into `configs/experiments/E006_yolo_seaslug.yaml` on the local PC (D-030 has the text),
commit, push, set `COMMIT` to that commit and run again from cell 1.

In [ ]:
# 1. Settings -- set COMMIT (full 40-character sha of the commit to run); keep everything else.
import os, re, subprocess, sys

REPO = "SakshamPainuly/underwater-creature-tracking"
BRANCH = "p4/colab-baseline"  # before PR #16 is merged: "anu/sea-slug-detection"
COMMIT = "0000000000000000000000000000000000000000"  # REQUIRED: `git rev-parse HEAD` on the local PC, after that commit is pushed
DATASET_VERSION = "v3"                                         # frozen Tritonia version (D-030)
EXPERIMENT_CONFIG = "configs/experiments/E006_yolo_seaslug.yaml"
EXPERIMENT_ID = "E006"                                         # must match the config
TOKEN_SECRET_NAME = "UWCT_GITHUB_TOKEN"  # Colab Secrets: see docs/COLAB.md (D-021)

REPO_DIR = "/content/uwct"                                                   # fixed: resumed runs expect the same dataset path
DRIVE_BOXES = "/content/drive/MyDrive/uwct/raw/tritonia_boxes_batch1"        # label source (read-only)
DRIVE_FRAMES = "/content/drive/MyDrive/uwct/work/tritonia_label_batch1/images"  # the 150 clean frames
RUNS_ROOT = "/content/drive/MyDrive/uwct/runs"                               # persistent run folders
LOCAL_BOXES = "/content/raw/tritonia_boxes_batch1"                           # fast local copies
LOCAL_FRAMES = "/content/tritonia_frames"
WEIGHTS_DIR = "/content/weights"                                             # pretrained checkpoints

assert re.fullmatch(r"[0-9a-f]{40}", COMMIT) and set(COMMIT) != {"0"}, "Set COMMIT to the exact 40-character commit sha to run"


def sh(*cmd):
    """Run a command in the repo, stream its output, raise if it fails."""
    p = subprocess.Popen(list(cmd), cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {' '.join(cmd)}")

In [ ]:
# 2. GPU present?
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU -- switch the runtime to GPU")

In [ ]:
# 3. Mount Google Drive (Tritonia frames + labels, persistent runs folder)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUNS_ROOT, exist_ok=True)

In [ ]:
# 4. Check out the exact commit (read-only token from Colab Secrets; see docs/COLAB.md).
#    Git asks the GIT_ASKPASS helper for credentials; the helper answers from the environment of
#    that one git process. The token is read from Colab Secrets inside git_remote() at call time
#    and is never in a command-line argument, URL, file, .git/config or output.
import stat
from google.colab import userdata

REPO_URL = f"https://github.com/{REPO}.git"  # no credentials in the URL
ASKPASS = "/content/uwct_git_askpass.sh"      # holds no secret; printf/echo are shell builtins
with open(ASKPASS, "w") as f:
    f.write('#!/bin/sh\n'
            'case "$1" in\n'
            '  Username*) echo "x-access-token" ;;\n'
            '  *) printf "%s\\n" "$UWCT_GIT_TOKEN" ;;\n'
            'esac\n')
os.chmod(ASKPASS, stat.S_IRWXU)


def git(*args, cwd=REPO_DIR):
    """Local git commands: no credentials involved."""
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{r.stderr}")
    return r.stdout.strip()


def git_remote(*args, cwd=REPO_DIR):
    """clone/fetch: the token exists only in this git child's environment, for this call only."""
    env = {**os.environ, "GIT_ASKPASS": ASKPASS, "GIT_TERMINAL_PROMPT": "0",
           "UWCT_GIT_TOKEN": userdata.get(TOKEN_SECRET_NAME)}
    try:
        # credential.helper= (empty) disables every credential helper, so nothing is stored
        r = subprocess.run(["git", "-c", "credential.helper=", *args], cwd=cwd, env=env,
                           capture_output=True, text=True)
        err = r.stderr.replace(env["UWCT_GIT_TOKEN"], "***")
        config = os.path.join(REPO_DIR, ".git", "config")
        leaked = os.path.isfile(config) and env["UWCT_GIT_TOKEN"] in open(config).read()
    finally:
        env.clear()
    if r.returncode != 0:
        if "403" in err or "not granted" in err:  # token is valid but cannot read this repo
            err += (f"\nThe token in Colab Secret {TOKEN_SECRET_NAME} has no read access to {REPO}."
                    " A fine-grained token only reaches repos owned by its resource owner, so a"
                    " collaborator's token cannot read this personal repo: the repo owner must create"
                    " it (resource owner = repo owner, only this repo, Contents: Read-only)."
                    " See docs/COLAB.md, 'One-time setup'.")
        raise RuntimeError(f"git {args[0]} failed:\n{err}")
    assert not leaked, "token found in .git/config"


if not os.path.isdir(REPO_DIR):
    git_remote("clone", "--branch", BRANCH, REPO_URL, REPO_DIR, cwd="/content")
else:  # reconnect / new commit: fetch the branch again
    git_remote("fetch", "origin", BRANCH)
git("checkout", "--detach", COMMIT)
assert git("rev-parse", "HEAD") == COMMIT
assert git("remote", "get-url", "origin") == REPO_URL
print("commit:", COMMIT, "| working tree:", git("status", "--porcelain") or "clean")

In [ ]:
# 5. Install the pinned GPU requirements (ultralytics==8.4.155; torch/CUDA come from Colab)
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-gpu.txt")
sh(sys.executable, "-c", "import ultralytics, torch; print('ultralytics', ultralytics.__version__, "
   "'| torch', torch.__version__, '| cuda', torch.version.cuda)")

In [ ]:
# 6. Environment report for this session (each run record stores its own copy too)
sh(sys.executable, "scripts/env_report.py", "--out", "/content/env_report_session.json")

In [ ]:
# 7. Copy the Tritonia frames and label source from Drive to local disk. Nothing is written to Drive.
#    The label source is registered as a read-only raw root (UWCT_TRITONIA_BOXES_ROOT): scripts refuse
#    to write inside it.
import shutil
for src, dst in ((DRIVE_BOXES, LOCAL_BOXES), (DRIVE_FRAMES, LOCAL_FRAMES)):
    assert os.path.isdir(src), f"{src} not found on Drive -- upload it first (see the top of this notebook)"
    if os.path.isdir(dst):
        shutil.rmtree(dst)  # start clean
    shutil.copytree(src, dst)
os.environ["UWCT_TRITONIA_BOXES_ROOT"] = LOCAL_BOXES  # read by every later script
print(len(os.listdir(os.path.join(LOCAL_BOXES, "labels"))), "label files |",
      len([f for f in os.listdir(LOCAL_FRAMES) if f.endswith(".png")]), "frames")

In [ ]:
# 8. Raw-data check: the frozen v3 manifests record the sha256 of every frame and label file, and the
#    label source's ingest_report.json. Cell 9 refuses to build if any of them differs, so the build is
#    the content check of this local copy (CLAUDE.md section 6).
import json
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from dataset_audit.versioning import read_version_meta, verify_frozen
verify_frozen(DATASET_VERSION)
meta = read_version_meta(DATASET_VERSION)
assert meta["dataset_kind"] == "tritonia", meta
print(json.dumps({k: meta[k] for k in ("name", "decision", "status", "class_names", "split_rule")}, indent=2))

In [ ]:
# 9. Build the YOLO view of v3 (copies, every file hash-checked) and print its dataset fingerprint.
sh(sys.executable, "scripts/prepare_tritonia_yolo.py", "--version", DATASET_VERSION,
   "--frames", LOCAL_FRAMES, "--boxes", LOCAL_BOXES)
DATASET_DIR = f"data/yolo/tritonia_{DATASET_VERSION}"
BUILT_FINGERPRINT = json.load(open(os.path.join(REPO_DIR, DATASET_DIR, "build_summary.json")))["dataset_fingerprint"]
print("dataset_fingerprint for E006:", BUILT_FINGERPRINT)

In [ ]:
# 10. Ultralytics reads the view exactly as built (names {0: lobster, 1: sea_slug}). Must print VERIFIED.
sh(sys.executable, "scripts/verify_yolo_dataset.py", "--dataset", DATASET_DIR)

In [ ]:
# 11. The experiment config exists at COMMIT and names this dataset and fingerprint.
from training.experiment import load_config
config_path = os.path.join(REPO_DIR, EXPERIMENT_CONFIG)
assert os.path.isfile(config_path), (f"{EXPERIMENT_CONFIG} is not in this commit yet. Create it on the local PC "
                                     f"with dataset_fingerprint: \"{BUILT_FINGERPRINT}\" (template in D-030), "
                                     "commit, push, set COMMIT and start again from cell 1.")
cfg = load_config(config_path)
assert cfg["id"] == EXPERIMENT_ID and cfg["dataset_version"] == DATASET_VERSION, cfg
assert cfg["dataset_fingerprint"] == BUILT_FINGERPRINT, (cfg["dataset_fingerprint"], BUILT_FINGERPRINT)
print("config OK:", cfg["id"], cfg["model"], cfg["train_args"])

In [ ]:
# 12. Preflight: every training check + load (download) the exact pretrained checkpoint.
TRAIN = [sys.executable, "scripts/train_yolo.py", "--config", EXPERIMENT_CONFIG,
         "--runs-root", RUNS_ROOT, "--weights-dir", WEIGHTS_DIR]
sh(*TRAIN, "--preflight-only")

In [ ]:
# 13. Smoke test: 1 epoch on 10% of train, validated on val -> RUNS_ROOT/E006_smoke_<utc>/
RUN_SMOKE = False  # set to True deliberately
assert RUN_SMOKE, "Smoke test not enabled"
sh(*TRAIN, "--smoke")

In [ ]:
# 14. Full training run -> RUNS_ROOT/E006/. Only after the smoke test passed and the run is approved.
RUN_FULL = False  # set to True deliberately
assert RUN_FULL, "Full training not enabled"
sh(*TRAIN)

In [ ]:
# 15. Resume after a Colab disconnect: re-run cells 1-11 (same COMMIT), then this cell.
RESUME = False  # set to True deliberately
assert RESUME, "Resume not enabled"
sh(*TRAIN, "--resume")

In [ ]:
# 16. Summary of the run record. Then download run_record.json and, on the local PC:
#     python scripts/register_run.py <path>/run_record.json
rec = json.load(open(os.path.join(RUNS_ROOT, EXPERIMENT_ID, "run_record.json"), encoding="utf-8"))
print(json.dumps({k: rec[k] for k in ("run_name", "git", "seed", "ultralytics_version", "val_metrics")}, indent=2))
print("dataset:", rec["dataset"]["version"], rec["dataset"]["fingerprint"])
print("best:", rec["outputs"]["best"])

In [ ]:
# 17. Val point scores against the MTrackJ crosses (D-031): the accuracy of E006. Val only.
#     Runs best.pt on every frame of the 3 val days (frames the automatic labels excluded too),
#     matches boxes to crosses (10 px, one to one) and chooses the threshold with the best val F1.
#     Writes new files in the run folder; refuses to overwrite an earlier result.
LOCAL_BATCH = "/content/tritonia_batch"
DRIVE_BATCH = os.path.dirname(DRIVE_FRAMES)
if os.path.isdir(LOCAL_BATCH):
    shutil.rmtree(LOCAL_BATCH)
shutil.copytree(DRIVE_BATCH, LOCAL_BATCH, ignore=shutil.ignore_patterns("hints"))
best = rec["outputs"]["best"]["path"]
run_dir = os.path.join(RUNS_ROOT, EXPERIMENT_ID)
sh(sys.executable, "scripts/tritonia_point_eval.py", "--batch", LOCAL_BATCH, "--split", "val",
   "--weights", best, "--imgsz", str(cfg["train_args"]["imgsz"]),
   "--pred-dir", os.path.join(run_dir, "val_point_predictions"),
   "--out", os.path.join(run_dir, "val_point_scores.json"))

## Not in this notebook yet

- The **final test evaluation**: test is scored only for a declared final candidate, with the
  threshold chosen on val (cell 17's `best_f1_on_val`), logged in `docs/EXPERIMENTS.md`:
  `scripts/tritonia_point_eval.py --split test --final E006 --conf <val threshold> ...`
  (CLAUDE.md rule 7; the script refuses test without `--final` and `--conf`).
- Report E006 with point precision, recall, F1 and point AP on val, overall and per day, never a
  single number. No mAP50/mAP50-95 for Tritonia: that needs hand-drawn boxes (D-031). Point
  precision is a lower bound (it is UNKNOWN whether every slug was clicked). v3 is 150 frames from
  one setup, so val and test measure new days, not new sites.